# Project 3: Linear Regression & Model Evaluation

**Dataset:** California Housing

**Goal:** Build linear regression models, evaluate them with proper diagnostics, and understand regularization (Ridge/Lasso).

In [ ]:
# ====== Setup ======
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120
DATA_URL = 'https://raw.githubusercontent.com/lucyyuhongxia-gmail/A4_ML_Projects/main/datasets'
def load_data(fn):
    df = pd.read_csv(f'{DATA_URL}/{fn}')
    print(f'Loaded: {fn} -> {df.shape[0]:,} rows x {df.shape[1]} cols')
    return df
print('Setup OK!')
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from scipy import stats

## 1. Simple Linear Regression — One Feature

In [ ]:
df = load_data('housing.csv')
X = df[['MedInc','HouseAge','AveRooms','AveBedrms','Population','AveOccup','Latitude','Longitude']]
y = df['MedHouseVal']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Simple: only MedInc (median income) predicts house value
model = LinearRegression().fit(X_train[['MedInc']], y_train)
y_pred = model.predict(X_test[['MedInc']])

print(f'Equation: HouseValue = {model.coef_[0]:.4f} x MedInc + {model.intercept_:.4f}')
print(f'R² (test):  {r2_score(y_test, y_pred):.4f}')
print(f'RMSE:       ${np.sqrt(mean_squared_error(y_test, y_pred))*100:.0f}K')

## 2. Multiple Linear Regression — All Features

In [ ]:
model_multi = LinearRegression().fit(X_train, y_train)
y_pred_multi = model_multi.predict(X_test)
r2 = r2_score(y_test, y_pred_multi)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_multi))
print(f'Multiple R²:  {r2:.4f} (vs Simple: 0.4734)')
print(f'  Improvement: +{(r2/0.4734-1)*100:.1f}%')
print(f'RMSE:         ${rmse*100:.0f}K (vs Simple: $83K)')
print(f'  Reduction:   -{(1-rmse/0.8310)*100:.1f}%')

## 3. Residual Diagnostics

In [ ]:
residuals = y_test - y_pred_multi
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals vs Predicted — check for patterns (should be random)
axes[0,0].scatter(y_pred_multi, residuals, alpha=0.2, s=5)
axes[0,0].axhline(y=0, color='r', ls='--'); axes[0,0].set_title('Residuals vs Predicted')

# 2. Histogram — should be roughly normal
axes[0,1].hist(residuals, bins=50, color='#2e86de', edgecolor='white', alpha=0.7)
axes[0,1].axvline(x=0, color='r', ls='--'); axes[0,1].set_title('Residual Distribution')

# 3. Q-Q plot — points should follow diagonal line
stats.probplot(residuals, dist='norm', plot=axes[1,0]); axes[1,0].set_title('Q-Q Plot')
axes[1,0].grid(True, alpha=0.3)

# 4. Actual vs Predicted — should cluster around y=x
axes[1,1].scatter(y_test, y_pred_multi, alpha=0.2, s=5)
axes[1,1].plot([0,5], [0,5], 'r--'); axes[1,1].set_title(f'Actual vs Predicted (R²={r2:.4f})')
plt.tight_layout(); plt.show()

## 4. Ridge & Lasso Regularization

In [ ]:
# Ridge adds L2 penalty (keeps all features, shrinks coefficients)
# Lasso adds L1 penalty (can zero out features entirely)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train); X_test_s = scaler.transform(X_test)

print(f'{"Alpha":8s}  {"Ridge R²":10s}  {"Lasso R²":10s}  {"Lasso Zeros"}')
for alpha in [0.01, 0.1, 1, 10, 100]:
    r2_r = Ridge(alpha=alpha).fit(X_train_s, y_train).score(X_test_s, y_test)
    lasso = Lasso(alpha=alpha).fit(X_train_s, y_train)
    r2_l = lasso.score(X_test_s, y_test)
    zeros = (np.abs(lasso.coef_) < 1e-10).sum()
    print(f'{alpha:8.2f}  {r2_r:10.4f}  {r2_l:10.4f}  {zeros:4d} features')

print()
print('💡 At alpha=100, Lasso kills ALL features (R² becomes negative!)')
print('   This is over-regularization — too much penalty.')

## Check Your Understanding
1. R²=0.58 means the model explains 58% of the variance in house prices. What about the remaining 42%?
2. What does a funnel-shaped residual plot indicate? How would you fix it?
3. Lasso at alpha=100 gives negative R². Is that possible? What does it mean?
4. Why does multiple regression beat simple regression, even though MedInc is the best single feature?